# Issue triage: train the models

This notebook repeats the training run from `docs/TRIAGE_PROTOCOL.md`. It has not been run on Colab. The reported results come from a run on a Mac with an Apple GPU; a Colab run needs its own numbers.

Upload the project folder (without `.venv*`, `data/raw` and `models`) to the Colab files area first, then run the cells in order.

In [ ]:
%cd /content/mlProject
!pip install -q -r requirements-train.txt

In [ ]:
import sys
sys.path.insert(0, 'src')
import torch
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('training device:', device)

## Data

Downloads the two files from the pinned commit and checks their hashes. The raw files stay out of git because the upstream repository declares no licence.

In [ ]:
from triage.cli import main
main(['fetch-data'])
main(['inventory'])

## Compare the models on the development set

This fits the baselines and the SetFit checkpoints, picks the best on development macro-F1 (a SetFit model must win by more than 0.02) and saves it to `models/candidate`.

In [ ]:
main(['train', '--setfit', 'sentence-transformers/all-MiniLM-L6-v2', 'sentence-transformers/all-MiniLM-L12-v2', '--iterations', '10', '--epochs', '1', '--device', device])

## Score the held-out test set once

The command refuses to run a second time for the same report folder.

In [ ]:
main(['final-test'])

In [ ]:
import json
print(json.dumps(json.load(open('reports/final_test.json'))['per_class'], indent=1))